<a href="https://colab.research.google.com/github/hsnheg1958-hub/Data-Analysis/blob/main/Bagging_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Complete Guide to Bagging (Bootstrap Aggregating)

Bagging is an ensemble meta-algorithm in machine learning designed to improve the stability and accuracy of machine learning algorithms.

## 1. Main Concepts and Terms

* **Ensemble Learning**: Combining predictions of multiple base models to produce a single robust model.
* **Bootstrap Sampling**: Randomly drawing subsets of data from the original dataset **with replacement**.
* **Aggregation**: Combining predictions from all models (majority voting for classification, averaging for regression).
* **Out-Of-Bag (OOB) Error**: Evaluating performance using training instances that were not selected during bootstrap sampling.

## 2. Main Goals and Usage

Bagging is primarily designed to address the key challenges of supervised learning models. Here is a clearer, more detailed breakdown of its goals and usage:

### A. Primary Goals
* **Reduce Variance (Overfitting)**: High-variance models (like unpruned Decision Trees) fit the training data extremely well but fail to generalize. Bagging reduces variance by averaging several highly randomized models, neutralizing individual model errors.
* **Increase Robustness and Stability**: Since each base model is trained on a slightly different subset of data, the ensemble becomes less sensitive to noise, outliers, and minor shifts in the training dataset.
* **Maintain Low Bias**: Because bagging does not alter how each base model is trained, the average model maintains a bias level similar to that of a single base model.

### B. When to Use Bagging
* **When Using "Unstable" Classifiers/Regressors**: An algorithm is unstable if small changes in the training data lead to large changes in the resulting predictor (e.g., Decision Trees, Neural Networks, Linear Regression with many collinear variables).
* **When You Want Out-Of-Bag (OOB) Evaluation**: Bagging allows you to test your model's performance on the fly without needing a separate validation dataset, since ~36.8% of the data is left unused for each base estimator.

### C. When NOT to Use Bagging
* **With Low-Variance Models (e.g., Linear Regression, Naive Bayes)**: If the base estimator has high bias and low variance, averaging their predictions will not yield any significant performance improvements and will only increase computational cost.

## 3. Practical Steps in Sequence

1. **Bootstrap**: Generate $N$ subsets with replacement.
2. **Train**: Build a separate base estimator on each subset.
3. **Aggregate**: Collect all individual outputs and compute the final mean/mode.

## 4. Practical Example for Classification
We will use `BaggingClassifier` with `DecisionTreeClassifier` on the **German Credit Dataset** (fetched from OpenML), which predicts whether a bank customer has good or bad credit risk.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# 1. Fetch German Credit Dataset ('credit-g')
print("Fetching German Credit dataset ('credit-g')...")
credit_data = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')
X_clf = credit_data.data
y_clf = credit_data.target

# Map target labels ('good'/'bad') to numeric 1 and 0
y_clf = y_clf.map({'good': 1, 'bad': 0})

# 2. Convert categorical features to numeric using One-Hot Encoding
X_clf = pd.get_dummies(X_clf, drop_first=True)

# Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

# 3. Initialize and train Bagging Classifier
bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=5, random_state=42),
    n_estimators=100,
    max_samples=0.8,
    oob_score=True,
    random_state=42
)
bag_clf.fit(X_train, y_train)

# 4. Evaluate
y_pred = bag_clf.predict(X_test)
print(f'\nOOB Score: {bag_clf.oob_score_:.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')

### 💡 Dataset & Feature Quick Hint

The **German Credit Dataset** (`credit-g`) consists of 1,000 bank customer records evaluated for credit risk. It contains 20 features detailing the financial and personal status of each applicant:

* **Target Variable (`class`)**:
  * `1` (Good Credit) — Low risk of defaulting.
  * `0` (Bad Credit) — High risk of defaulting.

* **Key Numerical Features**:
  * `duration`: Credit duration in months.
  * `credit_amount`: Total loan amount requested.
  * `age`: Age of the customer in years.
  * `installment_commitment`: Installment rate in percentage of disposable income.

* **Key Categorical Features** (one-hot encoded in our pipeline):
  * `checking_status`: Status of the customer's existing checking account.
  * `savings_status`: Status of the customer's savings account or bonds.
  * `employment`: Length of present employment.
  * `personal_status`: Personal status and sex.
  * `housing`: Housing status (rent, own, or for free).

## 5. Practical Example for Regression
We will use `BaggingRegressor` with `DecisionTreeRegressor` on a synthetic regression dataset.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# Generate dataset
X_reg, y_reg = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

# Initialize and train Bagging Regressor
bag_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=100,
    max_samples=0.8,
    oob_score=True,
    random_state=42
)
bag_reg.fit(X_train_r, y_train_r)

# Evaluate
y_pred_r = bag_reg.predict(X_test_r)
print(f'OOB Score (R^2): {bag_reg.oob_score_:.4f}')
print(f'Test MSE: {mean_squared_error(y_test_r, y_pred_r):.4f}')

## Feature Importance for Bagging Regressor

While a `BaggingRegressor` does not have a direct `feature_importances_` attribute, we can extract the feature importances from each of the individual base estimators (the 100 decision trees) and compute their average.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Extract feature importances from all estimators in the ensemble
importances = np.mean([tree.feature_importances_ for tree in bag_reg.estimators_], axis=0)

# Create a DataFrame for visualization
feature_names = [f'Feature {i}' for i in range(X_reg.shape[1])]
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, hue='Feature', palette='viridis', legend=False)
plt.title('Average Feature Importance across Bagging Regressor Estimators')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

## Comparing BaggingRegressor with RandomForestRegressor Feature Importances

Let's train a `RandomForestRegressor` using the same dataset and compare how the feature importances differ. While Bagging aggregates standard trees trained on random subsets of samples, Random Forests introduce additional randomness by selecting random subsets of features at each split.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Train Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train_r, y_train_r)

# Prepare comparison DataFrame
comparison_df = pd.DataFrame({
    'Feature': feature_names,
    'Bagging Regressor': importances,
    'Random Forest': rf_reg.feature_importances_
})

# Melt DataFrame for seaborn visualization
comparison_melted = comparison_df.melt(id_vars='Feature', var_name='Model', value_name='Importance')

# Plot
plt.figure(figsize=(12, 6))
sns.barplot(x='Feature', y='Importance', hue='Model', data=comparison_melted, palette='muted')
plt.title('Feature Importance Comparison: Bagging vs. Random Forest')
plt.xlabel('Features')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Appendix: Simplified Mathematical Aspects

### Bootstrap Probability
The probability of selecting a specific instance $x_i$ from a dataset of size $N$ in a single draw is $\frac{1}{N}$.
The probability of **not** selecting it in a single draw is $1 - \frac{1}{N}$.
For $N$ draws, the probability that a sample is never selected (Out-Of-Bag) is:
$$\lim_{N \to \infty} \left(1 - \frac{1}{N}\right)^N = \frac{1}{e} \approx 0.368$$
Thus, approximately **$63.2\%$** of unique samples are selected in each bootstrap training set, while **$36.8\%$** remain Out-Of-Bag.

### Variance Reduction (Regression)
If we average $B$ independent, identically distributed predictors each with variance $\sigma^2$, the variance of the average predictor is reduced to:
$$\text{Var}\left(\frac{1}{B}\sum_{i=1}^{B} f_i(x)\right) = \frac{\sigma^2}{B}$$
In practice, because our datasets overlap, the predictors are correlated. If the pairwise correlation is $\rho$, the variance is:
$$\rho \sigma^2 + \frac{1-\rho}{B}\sigma^2$$